# Comparing BM25 with TF-IDF

## Step 1:- Loading the data

In [1]:
import json
import csv
import os

# Define file paths
corpus_path = "scifact/corpus.jsonl"
queries_path = "scifact/queries.jsonl"
qrels_train_path = "scifact/qrels/train.tsv"
qrels_test_path = "scifact/qrels/test.tsv"

# 1. Load the Corpus (All documents)
corpus = {}
with open(corpus_path, 'r', encoding='utf-8') as f:
    for line in f:
        doc = json.loads(line)
        corpus[doc['_id']] = doc

# 2. Load the Queries (All search terms)
queries = {}
with open(queries_path, 'r', encoding='utf-8') as f:
    for line in f:
        query = json.loads(line)
        queries[query['_id']] = query['text']

# 3. Helper function to load a qrels file
def load_qrels(file_path):
    qrels_dict = {}
    if not os.path.exists(file_path):
        print(f"Warning: {file_path} not found.")
        return qrels_dict
        
    with open(file_path, 'r', encoding='utf-8') as f:
        reader = csv.reader(f, delimiter='\t')
        next(reader) # Skip the header row (query-id, corpus-id, score)
        for row in reader:
            query_id, corpus_id, score = row
            if query_id not in qrels_dict:
                qrels_dict[query_id] = {}
            qrels_dict[query_id][corpus_id] = int(score)
    return qrels_dict

# Load train and test qrels separately
qrels_train = load_qrels(qrels_train_path)
qrels_test = load_qrels(qrels_test_path)

# Combine them into a single master evaluation dictionary
qrels_all = {**qrels_train, **qrels_test}

# Print summary diagnostics
print(f"--- Dataset Statistics ---")
print(f"Total documents loaded in Corpus : {len(corpus)}")
print(f"Total queries loaded            : {len(queries)}")
print(f"Queries with TRAIN answer keys  : {len(qrels_train)}")
print(f"Queries with TEST answer keys   : {len(qrels_test)}")
print(f"Total queries with answer keys  : {len(qrels_all)}")

--- Dataset Statistics ---
Total documents loaded in Corpus : 5183
Total queries loaded            : 1109
Queries with TRAIN answer keys  : 809
Queries with TEST answer keys   : 300
Total queries with answer keys  : 1109


## Step 2:- Text Preprocessing

This step involves:
* **Lowercasing:** 
* **Removing Punctuation**
* **Tokenization:** 

In [2]:
import re

# 1. Define our text cleaning pipeline
def preprocess_text(text):
    if not text:
        return []
        
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    tokens = text.split()
    
    return tokens

# 2. Process the Corpus
tokenized_corpus = {}
for doc_id, doc in corpus.items():
    full_text = doc.get('title', '') + " " + doc.get('text', '')
    tokenized_corpus[doc_id] = preprocess_text(full_text)

# 3. Process the Queries
tokenized_queries = {}
for query_id, text in queries.items():
    tokenized_queries[query_id] = preprocess_text(text)

# 4. Verify the results
sample_query_id = list(queries.keys())[0]
print("--- Preprocessing Verification ---")
print("ORIGINAL QUERY:")
print(queries[sample_query_id])
print("\nTOKENIZED QUERY:")
print(tokenized_queries[sample_query_id])

--- Preprocessing Verification ---
ORIGINAL QUERY:
0-dimensional biomaterials lack inductive properties.

TOKENIZED QUERY:
['0dimensional', 'biomaterials', 'lack', 'inductive', 'properties']


## Step 3:- Implementing baseline TF-IDF method

The baseline retrieval system uses a pure, additive **Term Frequency-Inverse Document Frequency (TF-IDF)** model. Instead of treating documents as geometric vectors with angles, this approach calculates a direct relevance score by summing up the raw weights of matching keywords.

* **Term Frequency (TF):** Measures how many times a query word appears in a specific document. Higher occurrences signal stronger relevance.
* **Inverse Document Frequency (IDF):** Measures how rare a word is across the entire corpus. Universal words receive lower weight, while rare, highly descriptive scientific terms receive massive weight.
* **Scoring Logic:** For each query, we bypass length normalization (`norm=None`) and sum up the exact $TF \times IDF$ values for all matching terms. The documents are then sorted in descending order to output the top 10 matches.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Prepare corpus strings and track their IDs in an ordered list
corpus_ids = list(tokenized_corpus.keys())
corpus_strings = [" ".join(tokens) for tokens in tokenized_corpus.values()]

print("Step 1: Initializing and fitting TF-IDF Vectorizer...")

vectorizer = TfidfVectorizer(norm=None)
X_corpus = vectorizer.fit_transform(corpus_strings)
vocab = vectorizer.vocabulary_
print(f"Done! Corpus matrix shape: {X_corpus.shape}")

Step 1: Initializing and fitting TF-IDF Vectorizer...
Done! Corpus matrix shape: (5183, 50625)


In [4]:
import numpy as np

# 2. Dictionary to store the top 10 ranked document IDs for each query
tfidf_rankings = {}

print("\nStep 2: Calculating raw TF*IDF scores for all queries...")
for query_id in qrels_all.keys():
    query_tokens = tokenized_queries[query_id]
    
    # Initialize an array of zeros (one for each document)
    scores = np.zeros(len(corpus_ids))
    
    # For every word in the query, if it exists in our dataset's vocabulary,
    # grab its TF*IDF column from the matrix and add it to our total scores.
    for word in query_tokens:
        if word in vocab:
            col_idx = vocab[word]
            # Extract the column and add it to the running total
            word_scores = X_corpus[:, col_idx].toarray().flatten()
            scores += word_scores
    
    # Sort the final summed scores in descending order
    top_indices = np.argsort(scores)[-10:][::-1]
    
    # Map the indices back to their true document IDs and pair them with their scores
    ranked_docs = []
    for idx in top_indices:
        doc_id = corpus_ids[idx]
        doc_score = scores[idx]
        ranked_docs.append((doc_id, doc_score))
        
    # Save the ranked list for this query ID
    tfidf_rankings[query_id] = ranked_docs

print(f"Done! Processed raw TF*IDF rankings for all {len(tfidf_rankings)} queries.")


Step 2: Calculating raw TF*IDF scores for all queries...
Done! Processed raw TF*IDF rankings for all 1109 queries.


In [5]:
# 3. Look at a quick sample of the final sorted rankings
sample_id = list(tfidf_rankings.keys())[0]
sample_query_text = queries[sample_id] # Grab the original query text

print(f"\n--- Verification: Top 3 Sorted Results for Query ID: {sample_id} ---")
print(f"Query Text: {sample_query_text}\n")

# Loop through the top 3 ranked documents for this query
for rank, (doc_id, score) in enumerate(tfidf_rankings[sample_id][:3], 1):
    # Retrieve the title from our original corpus dictionary
    doc_title = corpus[doc_id].get('title', 'No Title Available')
    
    print(f"Rank {rank} | Doc ID: {doc_id} | Raw TF*IDF Score: {score:.4f}")
    print(f"Title: {doc_title}")
    print("-" * 60)


--- Verification: Top 3 Sorted Results for Query ID: 0 ---
Query Text: 0-dimensional biomaterials lack inductive properties.

Rank 1 | Doc ID: 42731834 | Raw TF*IDF Score: 22.1172
Title: Lack of Absent in Melanoma 2 (AIM2) expression in tumor cells is closely associated with poor survival in colorectal cancer patients.
------------------------------------------------------------
Rank 2 | Doc ID: 3845894 | Raw TF*IDF Score: 20.9837
Title: Computational and Statistical Analyses of Amino Acid Usage and Physico-Chemical Properties of the Twelve Late Embryogenesis Abundant Protein Classes
------------------------------------------------------------
Rank 3 | Doc ID: 6517267 | Raw TF*IDF Score: 17.6937
Title: Barriers and facilitators to implement shared decision making in multidisciplinary sciatica care: a qualitative study
------------------------------------------------------------


## Step 4:- Implementing BM25 method

**BM25 (Best Matching 25)** is a sophisticated, state-of-the-art probabilistic ranking function that improves upon basic TF-IDF by addressing its core limitations. 

BM25 introduces two critical modifications:
1.  **Term Frequency Saturation ($k_1$):** In pure TF-IDF, a word appearing 20 times makes a document sound 20 times more relevant. BM25 dampens this; after a word appears a few times, its impact curve flattens out (saturates), preventing keyword stuffing from dominating results.
2.  **Document Length Normalization ($b$):** Long documents naturally contain more words, which gives them an unfair advantage in raw word counts. BM25 penalizes unusually long documents and rewards short, concise documents that hit the exact target keywords.

The index is built using `rank_bm25`, and documents are sorted in descending order to generate the top 10 predictions.

# BM25 Ranking Function

The BM25 score of a document \(D\) for a query \(Q\) is:

$$
BM25(Q,D)=\sum_{t\in Q}
IDF(t)
\cdot
\frac{f(t,D)(k_1+1)}
{f(t,D)+k_1\left(1-b+b\frac{|D|}{avgdl}\right)}
$$

where:

- \(f(t,D)\) = frequency of term \(t\) in document \(D\)
- \(|D|\) = length of document \(D\)
- \(avgdl\) = average document length in the collection
- \(k_1\) = term frequency saturation parameter
- \(b\) = document length normalization parameter

The inverse document frequency is:

$$
IDF(t)=
\log\left(
\frac{N-n_t+0.5}
{n_t+0.5}
+1
\right)
$$

where:

- \(N\) = total number of documents
- \(n_t\) = number of documents containing term \(t\)

Typical parameter values:

$$
k_1 = 1.5
$$

$$
b = 0.75
$$

In [6]:
from rank_bm25 import BM25Okapi

corpus_ids = list(tokenized_corpus.keys())
corpus_tokens = list(tokenized_corpus.values())

print("Step 1: Initializing BM25 Index...")
# Initialize the model by passing in the list of tokenized documents
bm25 = BM25Okapi(corpus_tokens)
print("Done! BM25 index built successfully.")

Step 1: Initializing BM25 Index...
Done! BM25 index built successfully.


In [7]:
import numpy as np

bm25_rankings = {}

print("\nStep 2: Calculating BM25 scores for all queries...")
for query_id in qrels_all.keys():
    query_tokens = tokenized_queries[query_id]
    
    # BM25Okapi automatically compares the query against all documents and returns a flat array of scores in the exact order of corpus_ids
    scores = bm25.get_scores(query_tokens)
    
    # Sort the scores in descending order and get the top 10 indices
    top_indices = np.argsort(scores)[-10:][::-1]
    
    # Map the indices back to their true document IDs and pair them with their scores
    ranked_docs = []
    for idx in top_indices:
        doc_id = corpus_ids[idx]
        doc_score = scores[idx]
        ranked_docs.append((doc_id, doc_score))
        
    # Save the ranked list for this query ID
    bm25_rankings[query_id] = ranked_docs

print(f"Done! Processed BM25 rankings for all {len(bm25_rankings)} queries.")


Step 2: Calculating BM25 scores for all queries...
Done! Processed BM25 rankings for all 1109 queries.


In [8]:
# 3. Look at a quick sample of the final sorted rankings
sample_id = list(bm25_rankings.keys())[0]
sample_query_text = queries[sample_id]

print(f"\n--- Verification: Top 3 Sorted Results for Query ID: {sample_id} ---")
print(f"Query Text: {sample_query_text}\n")

# Loop through the top 3 ranked documents for this query
for rank, (doc_id, score) in enumerate(bm25_rankings[sample_id][:3], 1):
    # Retrieve the title from our original corpus dictionary
    doc_title = corpus[doc_id].get('title', 'No Title Available')
    
    print(f"Rank {rank} | Doc ID: {doc_id} | BM25 Score: {score:.4f}")
    print(f"Title: {doc_title}")
    print("-" * 60)


--- Verification: Top 3 Sorted Results for Query ID: 0 ---
Query Text: 0-dimensional biomaterials lack inductive properties.

Rank 1 | Doc ID: 825728 | BM25 Score: 9.2200
Title: Metastatic colonization requires the repression of the epithelial-mesenchymal transition inducer Prrx1.
------------------------------------------------------------
Rank 2 | Doc ID: 4435369 | BM25 Score: 9.1065
Title: Extracellular vesicles for drug delivery.
------------------------------------------------------------
Rank 3 | Doc ID: 28138927 | BM25 Score: 8.9587
Title: Potent, Selective, and Orally Bioavailable Inhibitors of VPS34 Provide Chemical Tools to Modulate Autophagy in Vivo.
------------------------------------------------------------


## Step 5:- Evaluation

To determine which search engine performs better, we run both predicted top-10 lists against a human-curated answer key (`qrels_all`). We evaluate performance globally using three core metrics averaged across all queries:

1.  **Precision@10:** Measures precision/noise. Out of the 10 documents returned by the engine, what percentage are actually marked as relevant in the answer key? 
2.  **Recall@10:** Measures comprehensiveness/coverage. Out of all the relevant documents that exist for a query in the database, what percentage did our engine manage to catch within its top 10 slots?
3.  **NDCG@10 (Normalized Discounted Cumulative Gain):** Measures rank order quality. It evaluates whether the engine placed the absolute best, highly relevant documents at the very top (Ranks 1, 2, 3) rather than burying them down at Rank 10, penalizing poor placement logarithmically.

In [9]:
import math

def evaluate_rankings(rankings, qrels, k=10):
    total_queries = 0
    sum_precision = 0.0
    sum_recall = 0.0
    sum_ndcg = 0.0

    for query_id, true_docs in qrels.items():
        if query_id not in rankings:
            continue

        predicted_docs = [doc_id for doc_id, score in rankings[query_id][:k]]
        total_relevant = len(true_docs)

        if total_relevant == 0:
            continue

        total_queries += 1
        relevant_retrieved = 0
        dcg = 0.0

        # 1. Calculate Precision, Recall, and DCG
        for rank_idx, doc_id in enumerate(predicted_docs):
            if doc_id in true_docs and true_docs[doc_id] > 0:
                relevant_retrieved += 1
                relevance_score = true_docs[doc_id]
                
                # DCG Formula: Relevance / log2(Rank + 1)
                # rank_idx is 0-based, so rank_idx + 2 ensures Rank 1 divides by log2(2)
                dcg += relevance_score / math.log2(rank_idx + 2)

        precision_at_k = relevant_retrieved / k
        recall_at_k = relevant_retrieved / total_relevant

        # 2. Calculate Ideal DCG (IDCG)
        ideal_relevance_scores = sorted(true_docs.values(), reverse=True)
        idcg = 0.0
        for rank_idx, rel in enumerate(ideal_relevance_scores[:k]):
            idcg += rel / math.log2(rank_idx + 2)

        # NDCG is the ratio of what we got vs. what was ideally possible
        ndcg_at_k = dcg / idcg if idcg > 0 else 0.0

        sum_precision += precision_at_k
        sum_recall += recall_at_k
        sum_ndcg += ndcg_at_k

    return (sum_precision / total_queries), (sum_recall / total_queries), (sum_ndcg / total_queries)

print("Evaluating systems against ground truth...")

# Grade the TF-IDF Baseline
tfidf_p, tfidf_r, tfidf_n = evaluate_rankings(tfidf_rankings, qrels_all, k=10)

# Grade the BM25 Implementation
bm25_p, bm25_r, bm25_n = evaluate_rankings(bm25_rankings, qrels_all, k=10)

print("\n" + "="*50)
print(f"🏆 FINAL LEADERBOARD (Top 10 Results)")
print("="*50)
print(f"{'Metric':<15} | {'Raw TF-IDF':<15} | {'BM25':<15}")
print("-" * 50)
print(f"{'Precision@10':<15} | {tfidf_p:<15.4f} | {bm25_p:<15.4f}")
print(f"{'Recall@10':<15} | {tfidf_r:<15.4f} | {bm25_r:<15.4f}")
print(f"{'NDCG@10':<15} | {tfidf_n:<15.4f} | {bm25_n:<15.4f}")
print("="*50)

Evaluating systems against ground truth...

🏆 FINAL LEADERBOARD (Top 10 Results)
Metric          | Raw TF-IDF      | BM25           
--------------------------------------------------
Precision@10    | 0.0551          | 0.0846         
Recall@10       | 0.4943          | 0.7576         
NDCG@10         | 0.3200          | 0.6380         


## Optimization of the Hyper Parameters in BM25(k1,b)

- k1 = Controls the saturation point
- b  = Controls the amount of Normalization

In [10]:
import itertools
from rank_bm25 import BM25Okapi
import numpy as np

k1_values = [0.8, 1.2, 1.5, 2.0]
b_values = [0.4, 0.6, 0.75, 0.9]

best_ndcg = 0.0
best_params = ()

print("Running Grid Search...")

for k1, b in itertools.product(k1_values, b_values):
    
    # Initialize a fresh index with the current parameters
    bm25_tuned = BM25Okapi(corpus_tokens, k1=k1, b=b)
    
    # Generate the top 10 rankings for every query in the dataset
    tuned_rankings = {}
    for query_id in qrels_all.keys():
        scores = bm25_tuned.get_scores(tokenized_queries[query_id])
        top_indices = np.argsort(scores)[-10:][::-1]
        tuned_rankings[query_id] = [(corpus_ids[idx], scores[idx]) for idx in top_indices]
        
    # Grade this specific combination using your custom evaluate_rankings function
    _, _, ndcg = evaluate_rankings(tuned_rankings, qrels_all, k=10)
    
    print(f"Tested k1={k1:<4} | b={b:<4} --> NDCG@10: {ndcg:.4f}")
    
    # Save the highest score
    if ndcg > best_ndcg:
        best_ndcg = ndcg
        best_params = (k1, b)

print("\n" + "="*50)
print(f"🏆 Optimal Parameters Found: k1={best_params[0]}, b={best_params[1]}")
print(f"Highest NDCG@10 achieved: {best_ndcg:.4f}")
print("="*50)

Running Grid Search...
Tested k1=0.8  | b=0.4  --> NDCG@10: 0.6333
Tested k1=0.8  | b=0.6  --> NDCG@10: 0.6367
Tested k1=0.8  | b=0.75 --> NDCG@10: 0.6371
Tested k1=0.8  | b=0.9  --> NDCG@10: 0.6406
Tested k1=1.2  | b=0.4  --> NDCG@10: 0.6324
Tested k1=1.2  | b=0.6  --> NDCG@10: 0.6396
Tested k1=1.2  | b=0.75 --> NDCG@10: 0.6393
Tested k1=1.2  | b=0.9  --> NDCG@10: 0.6376
Tested k1=1.5  | b=0.4  --> NDCG@10: 0.6351
Tested k1=1.5  | b=0.6  --> NDCG@10: 0.6394
Tested k1=1.5  | b=0.75 --> NDCG@10: 0.6380
Tested k1=1.5  | b=0.9  --> NDCG@10: 0.6372
Tested k1=2.0  | b=0.4  --> NDCG@10: 0.6316
Tested k1=2.0  | b=0.6  --> NDCG@10: 0.6392
Tested k1=2.0  | b=0.75 --> NDCG@10: 0.6390
Tested k1=2.0  | b=0.9  --> NDCG@10: 0.6373

🏆 Optimal Parameters Found: k1=0.8, b=0.9
Highest NDCG@10 achieved: 0.6406


## Final Conclusion: The TL;DR

1. **BM25 Wins:** It is vastly better than basic TF-IDF at finding the right scientific papers and putting them exactly where the user can see them (Rank 1-3).
2. **Why it Wins:**
 - ***Term Saturation***: TF-IDF is linear, but BM25 doesn't consider if the term is repeated many number of times
 - ***Length Normalization***: TF-IDF gives the same score for two documents where a term is repeated the same number of times, but one document is very large than the other. But BM25 avoids this by comparing the current document with the average length of a document.
3. **The Perfect Tune:** By testing different settings (k1=0.8, b=0.9), we told the algorithm to be extremely strict against long, wordy papers. This gave us the most accurate, focused search engine possible for this dataset!